# 06 · Ficha del Bloque 2 (Company Profile)

**Componente B · Company Profile, Bloque 2** · Crudo → **plata** (`plata/ficha_bloque2.csv`)

Arma, para cada empresa capturada, los tres campos del Bloque 2 a partir de lo que la propia empresa publica. Todo es **`literal`** (texto copiado de la fuente, recortado si hace falta); no se redacta nada.

| Campo | Regla | De dónde sale |
| --- | --- | --- |
| `tagline` | ≤ 150 caracteres | Descripción de la portada: JSON-LD `Organization` → `og:description` → `meta description`; si no hay, la primera oración de la descripción |
| `descripcion` | 200–1.200 caracteres, 2–3 párrafos | Texto principal (trafilatura) de las páginas "nosotros"; si no hay, de la portada |
| `productos_servicios` | Lista de 2–15 elementos | Títulos (`h2`–`h4`) de las páginas de servicios o productos, sin títulos de sección ni del menú |

Cada campo guarda la URL de su fuente. Se quitan correos, teléfonos y cargos directivos (LOPDP).

In [1]:
from pathlib import Path

ETL = Path.cwd().resolve()
if ETL.name == "notebooks":
    ETL = ETL.parent
DATOS = ETL / "datos"
CRUDO = DATOS / "crudo"      # bronce: datos tal como llegan, sin modificar
SCVS = CRUDO / "2025"        # descargas oficiales de la SCVS
WEB = CRUDO / "web"          # páginas capturadas de las webs oficiales
PLATA = DATOS / "plata"      # datos limpios y validados
ORO = DATOS / "oro"          # datos listos para publicar
for carpeta in (WEB, PLATA, ORO):
    carpeta.mkdir(parents=True, exist_ok=True)

In [2]:
import json
import re

import pandas as pd
import pandera.pandas as pa
import trafilatura
from scrapling.parser import Selector

pd.set_option("display.max_colwidth", 120)

## 1. Páginas capturadas

In [3]:
registros = [json.loads(l) for l in (WEB / "manifest.jsonl").read_text(encoding="utf-8").splitlines() if l]
paginas = pd.DataFrame(registros).drop_duplicates(["expediente", "sha256"])
paginas = paginas[paginas["status"] == 200]

def html_de(fila) -> str:
    return (WEB / f"{fila['expediente']}_{fila['sha256']}.html").read_text(encoding="utf-8")

paginas[["expediente", "tipo_pagina", "url"]]

,expediente,tipo_pagina,url
0,384,portada,https://www.corporacionfavorita.com/
1,7154,portada,https://pronaca.com/
2,124193,portada,https://minutocorp.com/
3,384,nosotros,https://www.corporacionfavorita.com/quienes-somos
4,724114,portada,https://fertisa.com/
5,384,servicios,https://www.corporacionfavorita.com/marcas/hanaska-servicios
6,7154,nosotros,https://pronaca.com/somos-pronaca/acerca-de-nosotros/
7,384,otra,https://www.corporacionfavorita.com/marcas#servicios
8,7154,nosotros,https://pronaca.com/somos-pronaca/nuestra-historia/
9,124193,otra,https://minutocorp.com/nuestro-adn/


## 2. Limpieza de texto y datos personales

In [4]:
CORREO = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
TELEFONO = re.compile(r"(\+?593|\b0)[\d\s-]{7,}\d")
CARGOS = re.compile(r"\b(gerente|presidente|director|directora|vicepresidente|ceo|cfo|fundador)\b", re.I)
# Títulos de secciones o del menú que no son un producto ni un servicio.
SECCIONES = re.compile(r"^(nuestr[oa]s? )?(servicios|productos|soluciones|contacto|cont[aá]ctanos|nosotros|qui[eé]nes somos|misi[oó]n|visi[oó]n|valores|historia|trayectoria|certificaciones|compromiso|m[eé]todo|noticias|blog|soporte|informaci[oó]n.*|trabaja con nosotros|s[ií]guenos|marcas propias|n[uú]mero.*)\b", re.I)
RUIDO = re.compile(r"cookies|derechos reservados|pol[ií]tica de privacidad|suscr[ií]bete|iniciar sesi[oó]n", re.I)

def limpiar(texto: str | None) -> str:
    if not isinstance(texto, str) or not texto:   # pandas trae NaN cuando falta el dato
        return ""
    texto = CORREO.sub("", TELEFONO.sub("", texto))
    return re.sub(r"\s+", " ", texto).strip()

def parrafos_utiles(texto: str) -> list[str]:
    salida = []
    for p in (texto or "").split("\n"):
        p = limpiar(p)
        if len(p) >= 60 and not CARGOS.search(p) and not RUIDO.search(p):
            salida.append(p)
    return salida

def recortar(texto: str, maximo: int) -> str:
    """Recorta en el último fin de oración que entra; si no hay, en la última palabra."""
    if len(texto) <= maximo:
        return texto
    corte = texto[:maximo]
    fin = max(corte.rfind(". "), corte.rfind("."))
    return corte[: fin + 1] if fin >= maximo // 2 else corte[: corte.rfind(" ")].rstrip(",;:") + "…"

## 3. Tagline

In [5]:
def tagline(empresa: pd.DataFrame) -> tuple[str, str]:
    portada = empresa[empresa["tipo_pagina"] == "portada"]
    for _, p in portada.iterrows():
        org = p["jsonld_organizacion"] if isinstance(p["jsonld_organizacion"], dict) else {}
        for candidato in (org.get("description"), p.get("og_description"), p.get("meta_description")):
            texto = limpiar(candidato)
            if len(texto) >= 20:
                return recortar(texto, 150), p["url"]
    return "", ""

## 4. Descripción

In [6]:
def descripcion(empresa: pd.DataFrame) -> tuple[str, str]:
    for tipo in ("nosotros", "portada"):
        for _, p in empresa[empresa["tipo_pagina"] == tipo].iterrows():
            texto = trafilatura.extract(html_de(p), include_comments=False, include_tables=False) or ""
            parrafos = parrafos_utiles(texto)
            if not parrafos:
                continue
            elegido, total = [], 0
            for par in parrafos:
                if total + len(par) > 1200 or len(elegido) == 3:
                    break
                elegido.append(par); total += len(par)
            resultado = "\n\n".join(elegido) if elegido else recortar(parrafos[0], 1200)
            if len(resultado) >= 200:
                return resultado, p["url"]
    return "", ""

## 5. Productos y servicios

In [7]:
def productos(empresa: pd.DataFrame) -> tuple[list[str], str]:
    candidatas = pd.concat([empresa[empresa["tipo_pagina"] == "servicios"], empresa[empresa["tipo_pagina"] == "otra"]])
    for _, p in candidatas.iterrows():
        pagina = Selector(html_de(p))
        vistos, lista = set(), []
        for t in pagina.css("main h2, main h3, main h4, article h2, article h3, h2, h3"):
            texto = limpiar(" ".join(t.css("::text").getall())).rstrip(":").strip()
            clave = texto.lower()
            if (3 <= len(texto) <= 60 and clave not in vistos and not SECCIONES.search(texto)
                    and not RUIDO.search(texto) and not CARGOS.search(texto)):
                vistos.add(clave); lista.append(texto)
        if len(lista) >= 2:
            return lista[:15], p["url"]
    return [], ""

## 6. Ficha por empresa

In [8]:
urls = pd.read_csv(PLATA / "urls_piloto.csv", dtype={"expediente": int})[["expediente", "nombre"]]
filas = []
for exp, empresa in paginas.groupby("expediente"):
    t, t_url = tagline(empresa)
    d, d_url = descripcion(empresa)
    if not t and d:   # sin descripción en la portada: primera oración de la descripción, si entra en 150
        primera = re.split(r"(?<=\.)\s", d, maxsplit=1)[0]
        if 20 <= len(primera) <= 150:
            t, t_url = primera, d_url
    ps, ps_url = productos(empresa)
    llenos = sum(bool(x) for x in (t, d, ps))
    filas.append({
        "expediente": exp,
        "tagline": t or None, "tagline_origen": "literal" if t else None, "tagline_fuente": t_url or None,
        "descripcion": d or None, "descripcion_fuente": d_url or None,
        "productos_servicios": "; ".join(ps) or None, "productos_fuente": ps_url or None,
        "fuente_tipo": "web_oficial",
        "fecha_captura": empresa["fecha_captura"].min()[:10],
        "estado": "completo" if llenos == 3 else ("parcial" if llenos else "pendiente"),
    })
ficha = urls.merge(pd.DataFrame(filas), on="expediente")
ficha[["nombre", "estado", "tagline", "productos_servicios"]]

,nombre,estado,tagline,productos_servicios
0,CORPORACION FAVORITA C.A.,completo,"Corporación Favorita, empresa ecuatoriana multinacional. Mejorando la calidad de vida cada día con retail, supermerc...",Supermercados y retail; Centros comerciales; Industria; Marcas de consumo; Construimos un gran lugar para trabajar.
1,PROCESADORA NACIONAL DE ALIMENTOS C.A. PRONACA,parcial,Somos una empresa multinacional familiar que actúa con respeto para alimentar bien a millones de familias en todo el...,NaN
2,FERTISA AGIF C.L.,completo,"Desde 1964, FERTISA AGIF ha liderado la importación y comercialización de fertilizantes e insumos agropecuarios de a...",Asesoría Técnica Personalizada; Transferencia de Tecnología; Fórmulas Personalizadas; Programas de Nutrición Vegetal...
3,MINUTOCORP S.A.,parcial,Minutocorp es una constructora y promotora inmobiliaria con más de 16 años aportando al desarrollo del Ecuador.,Construcción; Promoción Inmobiliaria; Build to Suit


## 7. Validación y guardado

In [9]:
esquema = pa.DataFrameSchema({
    "expediente": pa.Column(int, unique=True),
    "tagline": pa.Column(str, pa.Check.str_length(max_value=151), nullable=True),
    "descripcion": pa.Column(str, pa.Check.str_length(200, 1300), nullable=True),
    "tagline_origen": pa.Column(str, pa.Check.isin(["literal", "derivado"]), nullable=True),
    "estado": pa.Column(str, pa.Check.isin(["completo", "parcial", "pendiente"])),
}, strict=False)
ficha = esquema.validate(ficha, lazy=True)
ficha.to_csv(PLATA / "ficha_bloque2.csv", index=False)
ficha["estado"].value_counts()

estado
completo    2
parcial     2
Name: count, dtype: int64

## 8. Revisión: la ficha completa de cada empresa

In [10]:
for f in ficha.itertuples():
    print(f"■ {f.nombre}  [{f.estado}]")
    print(f"  Tagline: {f.tagline}")
    print(f"  Descripción ({0 if pd.isna(f.descripcion) else len(f.descripcion)} car.): {str(f.descripcion)[:400]}…")
    print(f"  Productos/servicios: {f.productos_servicios}\n")

■ CORPORACION FAVORITA C.A.  [completo]
  Tagline: Corporación Favorita, empresa ecuatoriana multinacional. Mejorando la calidad de vida cada día con retail, supermercados y compromiso con el…
  Descripción (313 car.): Trabajamos para mejorar la calidad de vida de todos, ofreciendo los mejores productos, servicios y experiencias, de forma eficiente, sostenible y responsable.

Ofrecemos productos y servicios de calidad y buscamos mejorar continuamente.

Creemos en nuestra gente, en su bienestar, felicidad y desarrollo integral.…
  Productos/servicios: Supermercados y retail; Centros comerciales; Industria; Marcas de consumo; Construimos un gran lugar para trabajar.

■ PROCESADORA NACIONAL DE ALIMENTOS C.A. PRONACA  [parcial]
  Tagline: Somos una empresa multinacional familiar que actúa con respeto para alimentar bien a millones de familias en todo el mundo.
  Descripción (653 car.): Somos una empresa multinacional familiar que actúa con respeto para alimentar bien a millones de familias